In [7]:
import pyodbc
import pandas as pd
import xgboost as xgb
import optuna
import shap
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from category_encoders.target_encoder import TargetEncoder


# 1. Connect to the database and fetch the data
conn = pyodbc.connect(
    r'DRIVER={ODBC Driver 17 for SQL Server};'
    r'SERVER=QUAN;'
    r'DATABASE=gt;'
    r'Trusted_Connection=yes;'
)

query = """
SELECT 
[call_type],
[priority],
[initial_call_type],
cast([cad_event_original_time_queued_date] as date) as  cad_event_original_time_queued_date,
cast([cad_event_original_time_queued_datetime_hour] as float) as cad_event_original_time_queued_datetime_hour,
[dispatch_precinct],
[dispatch_sector],
[dispatch_beat],
[dispatch_reporting_area],
[cad_event_response_category],
[call_type_indicator],
[dispatch_neighborhood],
[call_type_received_classification],
[call_sign_total_service_time_s],
dispatch_address,
(
SELECT COUNT(*) 
FROM [gt].[dbo].[call_data_20251019_processed_v44] t2
WHERE t2.cad_event_original_time_queued_datetime >= DATEADD(MINUTE, -60, t1.cad_event_original_time_queued_datetime)
AND t2.cad_event_original_time_queued_datetime <= t1.cad_event_original_time_queued_datetime
) AS previous_60_minute_count,

CASE 
WHEN DATENAME(WEEKDAY, cad_event_original_time_queued_datetime) IN ('Saturday', 'Sunday') THEN 'Weekend'
ELSE 'Weekday'
END AS DayType,

CASE 
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('January') THEN 1.081431401
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('February') THEN 1.050983984
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('March') THEN 1.058499074
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('April') THEN 1.007165803
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('May') THEN 1.09817936
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('June') THEN 0.934391305
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('July') THEN 0.926871054
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('August') THEN 0.911033386
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('September') THEN 0.972075484
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('October') THEN 0.993552683
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('November') THEN 0.987457854
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('December') THEN 0.978358612
ELSE null
END AS month_index,

case when co_response_call_sign_total_service_time_s > 0 and care_call_sign_total_service_time_s > 0 then 'Co-Response and Care-Call'
when co_response_call_sign_total_service_time_s > 0 then 'Co-Response'
when care_call_sign_total_service_time_s > 0 then 'Care-Call'
else NULL end as response_flags,


(
SELECT sum(cast(count_of_officers as int)) 
FROM [gt].[dbo].[call_data_20251019_processed_v44] t2
WHERE t2.cad_event_original_time_queued_datetime >= DATEADD(MINUTE, -60, t1.cad_event_original_time_queued_datetime)
AND t2.cad_event_original_time_queued_datetime <= t1.cad_event_original_time_queued_datetime
) AS previous_60_minute_officer_count

from [gt].[dbo].[call_data_20251019_processed_v44] t1
tablesample (10 percent)
"""

df = pd.read_sql(query, conn)
conn.close()

df = df.drop(columns=["cad_event_response_category"])

categorical_cols = [
    "call_type", "initial_call_type", "priority", "call_type_indicator", "dispatch_precinct",
    "dispatch_sector", "dispatch_beat", "call_type_indicator", "dispatch_reporting_area", "dispatch_neighborhood",
    "call_type_received_classification", "dispatch_address", "cad_event_original_time_queued_datetime_hour", "DayType",
    "response_flags"
]
for col in categorical_cols:
    df[col] = df[col].fillna("UNKNOWN")

#2. Feature Engineering
df["cad_event_original_time_queued_date"] = pd.to_datetime(df["cad_event_original_time_queued_date"])
df["day_of_week"] = df["cad_event_original_time_queued_date"].dt.dayofweek  # 0=Monday, 6=Sunday
df["month"] = df["cad_event_original_time_queued_date"].dt.month
df["hour"] = df["cad_event_original_time_queued_datetime_hour"].astype(int)

# Drop original datetime columns
df = df.drop(columns=["cad_event_original_time_queued_date", "cad_event_original_time_queued_datetime_hour"])

#3. Target Variable
# Log transform target to stabilize variance
df["log_call_sign_total_service_time_s"] = np.log1p(df["call_sign_total_service_time_s"])

# 4. Train-Test Split
X = df.drop(columns=["call_sign_total_service_time_s", "log_call_sign_total_service_time_s"])
y = df["log_call_sign_total_service_time_s"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#5. Column Transformer for Categorical/Continuous Features
# Define categorical and numerical features
categorical_features = [
    "call_type", "priority", "call_type_indicator", "dispatch_precinct",
    "dispatch_sector", "dispatch_reporting_area", "dispatch_neighborhood",
    "call_type_received_classification", "DayType","response_flags"
]
numerical_features = ["day_of_week", "month", "hour", "month_index", "previous_60_minute_officer_count", "previous_60_minute_count"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", TargetEncoder(), categorical_features),
        ("num", "passthrough", numerical_features)
    ])

# Fit and transform data
X_train_processed = preprocessor.fit_transform(X_train, y_train)
X_test_processed = preprocessor.transform(X_test)

# 6. Hyperparameter Tuning with Optuna
def objective(trial):
    params = {
        "n_estimators": 500,
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.5),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 10),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 10),
        "random_state": 42
    }
    model = xgb.XGBRegressor(**params)
    scores = cross_val_score(model, X_train_processed, y_train, cv=3, scoring="neg_mean_squared_error")
    return -scores.mean()

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)  # Increase trials for better tuning

best_params = study.best_params
print("Best Hyperparameters:", best_params)

#7. Train Final Model
final_model = xgb.XGBRegressor(**best_params)
final_model.fit(
    X_train_processed, y_train,
    eval_set=[(X_test_processed, y_test)],
    verbose=False
)

#8. Evaluate Model
y_pred = final_model.predict(X_test_processed)
y_true = df.loc[X_test.index, "call_sign_total_service_time_s"]
y_pred_exp = np.expm1(y_pred)  # Reverse log transformation

mse = mean_squared_error(y_true, y_pred_exp)
mae = mean_absolute_error(y_true, y_pred_exp)
r2 = r2_score(y_true, y_pred_exp)

print(f"MSE: {mse:.4f}, MAE: {mae:.4f}, R²: {r2:.4f}")

# --- 9. SHAP Analysis ---
#booster = final_model.get_booster()
#explainer = shap.Explainer(booster, X_train_processed)
#shap_values = explainer(X_test_processed)
#shap.summary_plot(shap_values, X_test_processed, feature_names=X.columns.tolist())

C:\Users\RQ\AppData\Local\Temp\ipykernel_13344\3900412579.py:84: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
[I 2025-11-13 18:44:02,379] A new study created in memory with name: no-name-f3b95d74-57c5-425b-80dc-73f6517ff2a5
[I 2025-11-13 18:44:02,918] Trial 0 finished with value: 2.4751639865539947 and parameters: {'max_depth': 3, 'learning_rate': 0.08412821664854586, 'subsample': 0.7011388190306822, 'colsample_bytree': 0.6381029798348191, 'reg_alpha': 2.773579377698727, 'reg_lambda': 2.00353418021212}. Best is trial 0 with value: 2.4751639865539947.
[I 2025-11-13 18:44:03,937] Trial 1 finished with value: 2.7026941211327 and parameters: {'max_depth': 6, 'learning_rate': 0.2007133798456397, 'subsample': 0.8476275460159084, 'colsample_bytree': 0.8685744552281727, 'reg_alpha': 2.7932071755191155, 'reg_l

Best Hyperparameters: {'max_depth': 5, 'learning_rate': 0.02156198401859405, 'subsample': 0.6457527407018471, 'colsample_bytree': 0.795608156456472, 'reg_alpha': 0.8636227123105205, 'reg_lambda': 1.3023177905017826}
MSE: 6863436.8694, MAE: 1725.4369, R²: -0.2516


In [16]:
import pandas as pd
import pyodbc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score

# Load data
conn = pyodbc.connect(
    r'DRIVER={ODBC Driver 17 for SQL Server};'
    r'SERVER=QUAN;'
    r'DATABASE=gt;'
    r'Trusted_Connection=yes;'
)
query = """
SELECT
[call_type],
[priority],
[initial_call_type],
cast([cad_event_original_time_queued_date] as date) as  cad_event_original_time_queued_date,
cast([cad_event_original_time_queued_datetime_hour] as float) as cad_event_original_time_queued_datetime_hour,
[dispatch_precinct],
[dispatch_sector],
[dispatch_beat],
[dispatch_reporting_area],
[cad_event_response_category],
[call_type_indicator],
[dispatch_neighborhood],
[call_type_received_classification],
dispatch_address,

(
SELECT COUNT(*) 
FROM [gt].[dbo].[call_data_20251019_processed_v44] t2
WHERE t2.cad_event_original_time_queued_datetime >= DATEADD(MINUTE, -60, t1.cad_event_original_time_queued_datetime)
AND t2.cad_event_original_time_queued_datetime <= t1.cad_event_original_time_queued_datetime
) AS previous_60_minute_count,

(
SELECT sum(cast(count_of_officers as int)) 
FROM [gt].[dbo].[call_data_20251019_processed_v44] t2
WHERE t2.cad_event_original_time_queued_datetime >= DATEADD(MINUTE, -60, t1.cad_event_original_time_queued_datetime)
AND t2.cad_event_original_time_queued_datetime <= t1.cad_event_original_time_queued_datetime
) AS previous_60_minute_officer_count,

CASE 
WHEN DATENAME(WEEKDAY, cad_event_original_time_queued_datetime) IN ('Saturday', 'Sunday') THEN 'Weekend'
ELSE 'Weekday'
END AS DayType,

CASE 
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('January') THEN 1.081431401
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('February') THEN 1.050983984
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('March') THEN 1.058499074
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('April') THEN 1.007165803
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('May') THEN 1.09817936
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('June') THEN 0.934391305

WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('July') THEN 0.926871054
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('August') THEN 0.911033386
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('September') THEN 0.972075484
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('October') THEN 0.993552683
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('November') THEN 0.987457854
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('December') THEN 0.978358612
ELSE null
END AS month_index,

case when co_response_call_sign_total_service_time_s > 0 and care_call_sign_total_service_time_s > 0 then 'Co-Response and Care-Call'
when co_response_call_sign_total_service_time_s > 0 then 'Co-Response'
when care_call_sign_total_service_time_s > 0 then 'Care-Call'
else NULL end as response_flags,

case when [call_sign_total_service_time_s] between 0 and 1200
then '0 - 20 min'
when [call_sign_total_service_time_s] between 1200 and 2400
then '21 - 40 min'
when [call_sign_total_service_time_s] between 2400 and 3600
then '41 - 60 min'
when [call_sign_total_service_time_s] >= 3600
then '60+ min'
else NULL end as service_time_flag

from [gt].[dbo].[call_data_20251019_processed_v44] t1
tablesample (10 percent)
"""
df = pd.read_sql(query, conn)
conn.close()

# Clean data
df = df.dropna(subset=['service_time_flag'])

# Encode target
le = LabelEncoder()
df['service_time_flag_encoded'] = le.fit_transform(df['service_time_flag'])

# Split data
X = df.drop(columns=['service_time_flag', 'service_time_flag_encoded'])
y = df['service_time_flag_encoded']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocess features
categorical_cols = X.select_dtypes(include=['object']).columns
numerical_cols = X.select_dtypes(include=['float64']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Train model
model = XGBClassifier(objective='multi:softmax', num_class=7, eval_metric='mlogloss')
model.fit(X_train_processed, y_train)

# Evaluate
y_pred = model.predict(X_test_processed)
print(classification_report(y_test, y_pred, target_names=le.classes_))
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

C:\Users\RQ\AppData\Local\Temp\ipykernel_17092\4047479688.py:104: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


              precision    recall  f1-score   support

  0 - 20 min       0.51      0.92      0.66      5477
 21 - 40 min       0.41      0.11      0.18      2350
 41 - 60 min       0.08      0.00      0.00      1315
     60+ min       0.46      0.20      0.28      2537

    accuracy                           0.50     11679
   macro avg       0.37      0.31      0.28     11679
weighted avg       0.43      0.50      0.40     11679

Accuracy: 0.4979


In [19]:
import pandas as pd
import pyodbc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score

# Load data
conn = pyodbc.connect(
    r'DRIVER={ODBC Driver 17 for SQL Server};'
    r'SERVER=QUAN;'
    r'DATABASE=gt;'
    r'Trusted_Connection=yes;'
)
query = """
SELECT
[call_type],
[priority],
[initial_call_type],
cast([cad_event_original_time_queued_date] as date) as  cad_event_original_time_queued_date,
cast([cad_event_original_time_queued_datetime_hour] as float) as cad_event_original_time_queued_datetime_hour,
[dispatch_precinct],
[dispatch_sector],
[dispatch_beat],
[dispatch_reporting_area],
[cad_event_response_category],
[call_type_indicator],
[dispatch_neighborhood],
[call_type_received_classification],
dispatch_address,

(
SELECT COUNT(*) 
FROM [gt].[dbo].[call_data_20251019_processed_v44] t2
WHERE t2.cad_event_original_time_queued_datetime >= DATEADD(MINUTE, -60, t1.cad_event_original_time_queued_datetime)
AND t2.cad_event_original_time_queued_datetime <= t1.cad_event_original_time_queued_datetime
) AS previous_60_minute_count,

(
SELECT sum(cast(count_of_officers as int)) 
FROM [gt].[dbo].[call_data_20251019_processed_v44] t2
WHERE t2.cad_event_original_time_queued_datetime >= DATEADD(MINUTE, -60, t1.cad_event_original_time_queued_datetime)
AND t2.cad_event_original_time_queued_datetime <= t1.cad_event_original_time_queued_datetime
) AS previous_60_minute_officer_count,

CASE 
WHEN DATENAME(WEEKDAY, cad_event_original_time_queued_datetime) IN ('Saturday', 'Sunday') THEN 'Weekend'
ELSE 'Weekday'
END AS DayType,

CASE 
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('January') THEN 1.081431401
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('February') THEN 1.050983984
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('March') THEN 1.058499074
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('April') THEN 1.007165803
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('May') THEN 1.09817936
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('June') THEN 0.934391305
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('July') THEN 0.926871054
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('August') THEN 0.911033386
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('September') THEN 0.972075484
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('October') THEN 0.993552683
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('November') THEN 0.987457854
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('December') THEN 0.978358612
ELSE null
END AS month_index,

case when co_response_call_sign_total_service_time_s > 0 and care_call_sign_total_service_time_s > 0 then 'Co-Response and Care-Call'
when co_response_call_sign_total_service_time_s > 0 then 'Co-Response'
when care_call_sign_total_service_time_s > 0 then 'Care-Call'
else NULL end as response_flags,

case when [call_sign_total_service_time_s] between 0 and 1800
then '0 - 30 min'
when [call_sign_total_service_time_s] between 1800 and 3600
then '30 - 60 min'
when [call_sign_total_service_time_s] >= 3600
then '60+ min'
else NULL end as service_time_flag

from [gt].[dbo].[call_data_20251019_processed_v44] t1
tablesample (10 percent)
"""
df = pd.read_sql(query, conn)
conn.close()

# Clean data
df = df.dropna(subset=['service_time_flag'])

# Encode target
le = LabelEncoder()
df['service_time_flag_encoded'] = le.fit_transform(df['service_time_flag'])

# Split data
X = df.drop(columns=['service_time_flag', 'service_time_flag_encoded'])
y = df['service_time_flag_encoded']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocess features
categorical_cols = X.select_dtypes(include=['object']).columns
numerical_cols = X.select_dtypes(include=['float64']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Train model
model = XGBClassifier(objective='multi:softmax', num_class=7, eval_metric='mlogloss')
model.fit(X_train_processed, y_train)

# Evaluate
y_pred = model.predict(X_test_processed)
print(classification_report(y_test, y_pred, target_names=le.classes_))
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

C:\Users\RQ\AppData\Local\Temp\ipykernel_17092\1962838087.py:102: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


              precision    recall  f1-score   support

  0 - 30 min       0.60      0.97      0.74      6886
 30 - 60 min       0.34      0.01      0.03      2395
     60+ min       0.50      0.11      0.19      2550

    accuracy                           0.59     11831
   macro avg       0.48      0.37      0.32     11831
weighted avg       0.53      0.59      0.48     11831

Accuracy: 0.5921


In [20]:
import pandas as pd
import pyodbc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score

# Load data
conn = pyodbc.connect(
    r'DRIVER={ODBC Driver 17 for SQL Server};'
    r'SERVER=QUAN;'
    r'DATABASE=gt;'
    r'Trusted_Connection=yes;'
)
query = """
SELECT
[call_type],
[priority],
[initial_call_type],
cast([cad_event_original_time_queued_date] as date) as  cad_event_original_time_queued_date,
cast([cad_event_original_time_queued_datetime_hour] as float) as cad_event_original_time_queued_datetime_hour,
[dispatch_precinct],
[dispatch_sector],
[dispatch_beat],
[dispatch_reporting_area],
[cad_event_response_category],
[call_type_indicator],
[dispatch_neighborhood],
[call_type_received_classification],
dispatch_address,

(
SELECT COUNT(*) 
FROM [gt].[dbo].[call_data_20251019_processed_v44] t2
WHERE t2.cad_event_original_time_queued_datetime >= DATEADD(MINUTE, -60, t1.cad_event_original_time_queued_datetime)
AND t2.cad_event_original_time_queued_datetime <= t1.cad_event_original_time_queued_datetime
) AS previous_60_minute_count,

(
SELECT sum(cast(count_of_officers as int)) 
FROM [gt].[dbo].[call_data_20251019_processed_v44] t2
WHERE t2.cad_event_original_time_queued_datetime >= DATEADD(MINUTE, -60, t1.cad_event_original_time_queued_datetime)
AND t2.cad_event_original_time_queued_datetime <= t1.cad_event_original_time_queued_datetime
) AS previous_60_minute_officer_count,

CASE 
WHEN DATENAME(WEEKDAY, cad_event_original_time_queued_datetime) IN ('Saturday', 'Sunday') THEN 'Weekend'
ELSE 'Weekday'
END AS DayType,

CASE 
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('January') THEN 1.081431401
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('February') THEN 1.050983984
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('March') THEN 1.058499074
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('April') THEN 1.007165803
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('May') THEN 1.09817936
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('June') THEN 0.934391305
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('July') THEN 0.926871054
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('August') THEN 0.911033386
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('September') THEN 0.972075484
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('October') THEN 0.993552683
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('November') THEN 0.987457854
WHEN DATENAME(month, cad_event_original_time_queued_datetime) IN ('December') THEN 0.978358612
ELSE null
END AS month_index,

case when co_response_call_sign_total_service_time_s > 0 and care_call_sign_total_service_time_s > 0 then 'Co-Response and Care-Call'
when co_response_call_sign_total_service_time_s > 0 then 'Co-Response'
when care_call_sign_total_service_time_s > 0 then 'Care-Call'
else NULL end as response_flags,




case when [call_sign_total_service_time_s] between 0 and 600
then '0 - 10 min'
when [call_sign_total_service_time_s] between 600 and 1200
then '11 - 20 min'
when [call_sign_total_service_time_s] between 1200 and 1800
then '21 - 30 min'
when [call_sign_total_service_time_s] between 1800 and 2400
then '31 - 40 min'
when [call_sign_total_service_time_s] between 2400 and 3000
then '41 - 50 min'
when [call_sign_total_service_time_s] between 3000 and 3600
then '51 - 60 min'
when [call_sign_total_service_time_s] >= 3600
then '60+ min'
else NULL end as service_time_flag
from [gt].[dbo].[call_data_20251019_processed_v44] t1
tablesample (10 percent)
"""
df = pd.read_sql(query, conn)
conn.close()

# Clean data
df = df.dropna(subset=['service_time_flag'])

# Encode target
le = LabelEncoder()
df['service_time_flag_encoded'] = le.fit_transform(df['service_time_flag'])

# Split data
X = df.drop(columns=['service_time_flag', 'service_time_flag_encoded'])
y = df['service_time_flag_encoded']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocess features
categorical_cols = X.select_dtypes(include=['object']).columns
numerical_cols = X.select_dtypes(include=['float64']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Train model
model = XGBClassifier(objective='multi:softmax', num_class=7, eval_metric='mlogloss')
model.fit(X_train_processed, y_train)

# Evaluate
y_pred = model.predict(X_test_processed)
print(classification_report(y_test, y_pred, target_names=le.classes_))
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

C:\Users\RQ\AppData\Local\Temp\ipykernel_17092\3697111370.py:95: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


              precision    recall  f1-score   support

  0 - 10 min       0.38      0.84      0.52      3832
 11 - 20 min       0.25      0.03      0.06      1804
 21 - 30 min       0.23      0.10      0.14      1317
 31 - 40 min       0.12      0.01      0.02      1036
 41 - 50 min       0.18      0.01      0.02       776
 51 - 60 min       0.00      0.00      0.00       609
     60+ min       0.36      0.35      0.35      2633

    accuracy                           0.36     12007
   macro avg       0.22      0.19      0.16     12007
weighted avg       0.29      0.36      0.27     12007

Accuracy: 0.3633
